In [1]:
pip install cartesia

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.7/210.7 kB 10.9 MB/s eta 0:00:00


In [3]:
import json
import os
from cartesia import Cartesia

# Initialize the client (ensure your API key is set safely)
client = Cartesia(api_key="sk_car_BsN7EN2Ctmv7C26JMsjq1U")

# Load your new evaluation dataset
input_json = "bengali_evaluation_set.json"
with open(input_json, "r", encoding="utf-8") as f:
    data = json.load(f)

output_dir = "sonic3_eval_audio_bengali"
os.makedirs(output_dir, exist_ok=True)

# Iterate through the list of dictionaries
for index, item in enumerate(data):
    # Grab the text using the new key
    text_to_speak = item.get("bengali_sentence", "")

    if not text_to_speak:
        continue

    print(f"Generating audio for item {index} (ID: {item.get('id')})...")

    # Call the Sonic 3 API
    response = client.tts.generate(
        model_id="sonic-3",
        transcript=text_to_speak,
        voice={"id": "2ba861ea-7cdc-43d1-8608-4045b5a41de5"}, # Your requested Voice ID
        language="bn", # Explicitly declare Bengali
        output_format={
            "container": "wav",
            "encoding": "pcm_s16le",
            "sample_rate": 44100
        }
    )

    # Save the output to a WAV file using Cartesia's built-in method
    output_filename = os.path.join(output_dir, f"eval_output_{index}.wav")
    response.write_to_file(output_filename)

print("Sonic 3 Bengali TTS generation complete!")

Generating audio for item 0 (ID: 1)...
Generating audio for item 1 (ID: 2)...
Generating audio for item 2 (ID: 3)...
Generating audio for item 3 (ID: 4)...
Generating audio for item 4 (ID: 5)...
Generating audio for item 5 (ID: 6)...
Generating audio for item 6 (ID: 7)...
Generating audio for item 7 (ID: 8)...
Generating audio for item 8 (ID: 9)...
Generating audio for item 9 (ID: 10)...
Generating audio for item 10 (ID: 11)...
Generating audio for item 11 (ID: 12)...
Generating audio for item 12 (ID: 13)...
Generating audio for item 13 (ID: 14)...
Generating audio for item 14 (ID: 15)...
Generating audio for item 15 (ID: 16)...
Generating audio for item 16 (ID: 17)...
Generating audio for item 17 (ID: 18)...
Generating audio for item 18 (ID: 19)...
Generating audio for item 19 (ID: 20)...
Sonic 3 Bengali TTS generation complete!


In [5]:
!pip install openai-whisper jiwer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 21.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 119.7 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=af1f70c5d035e90849aa1f5a0159eb4f38113c8bdd8628bf917132b643485893
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [6]:
import json
import os
import shutil
import whisper
from jiwer import wer, cer

# Load the Whisper Medium model
print("Loading Whisper 'medium' model...")
model = whisper.load_model("medium")

# Load your reference text from the JSON
input_json = "bengali_evaluation_set.json"
with open(input_json, "r", encoding="utf-8") as f:
    data = json.load(f)

audio_dir = "sonic3_eval_audio_bengali"
references = []
hypotheses = []

print("\nStarting Bengali transcription and evaluation...")

# Iterate through the JSON
for index, item in enumerate(data):
    reference_text = item.get("bengali_sentence", "")

    if not reference_text:
        continue

    audio_path = os.path.join(audio_dir, f"eval_output_{index}.wav")

    if not os.path.exists(audio_path):
        print(f"Skipping item {index} - Audio file not found.")
        continue

    # Transcribe the audio back to text (Forcing Bengali language)
    print(f"Transcribing {audio_path}...")
    result = model.transcribe(audio_path, language="bn")
    hypothesis_text = result["text"].strip()

    references.append(reference_text)
    hypotheses.append(hypothesis_text)

    print(f"Ref: {reference_text}")
    print(f"Hyp: {hypothesis_text}\n")

# Calculate WER and CER
if references and hypotheses:
    overall_wer = wer(references, hypotheses)
    overall_cer = cer(references, hypotheses)

    print("-" * 30)
    print("🏆 BENGALI EVALUATION RESULTS 🏆")
    print("-" * 30)
    print(f"Word Error Rate (WER):      {overall_wer:.4f} ({overall_wer * 100:.2f}%)")
    print(f"Character Error Rate (CER): {overall_cer:.4f} ({overall_cer * 100:.2f}%)")
    print("-" * 30)
else:
    print("No audio files were evaluated. Check your directory.")

# Zip the audio directory
zip_filename = "sonic3_bengali_eval_backup"
print(f"\nZipping folder '{audio_dir}' into '{zip_filename}.zip'...")
shutil.make_archive(zip_filename, 'zip', audio_dir)

# Trigger download for Colab/Jupyter
try:
    from google.colab import files
    print("Initiating download...")
    files.download(f"{zip_filename}.zip")
except ImportError:
    print(f"Successfully created {zip_filename}.zip in your current directory.")

Loading Whisper 'medium' model...


100%|█████████████████████████████████████| 1.42G/1.42G [00:17<00:00, 87.7MiB/s]



Starting Bengali transcription and evaluation...
Transcribing sonic3_eval_audio_bengali/eval_output_0.wav...
Ref: অন্ধকার ঘরে কাঁপা হাতে সে পুরোনো পুঁথি আর রথের ভাঙা চাকা খুঁজছে।
Hyp: अन्धकर गोरे कापा हाते शे पूरनो पूछी आ रथेर भागा चका खुन छे।

Transcribing sonic3_eval_audio_bengali/eval_output_1.wav...
Ref: ফাল্গুনের মেলায় ভণ্ড সাধুর কথায় ঘণ্টা বাজতেই এক অদ্ভুত কম্পন তৈরি হলো।
Hyp: ফালগনੁনিল মালা ভনੀডা শাথিਸ কাথা গੈনੈটা বাজੁয এক অদੱভੀত কানੀত তসিলੋ হੂইলੇ হੋইলੋকানੋ ওੀনੀੀনੀলੀ উলੇੇ ওলੀকালੀন�

Transcribing sonic3_eval_audio_bengali/eval_output_2.wav...
Ref: শহরের এই উচ্চ অট্টালিকার ছাদে দাঁড়ালে উদ্দাম বাতাসের শব্দে অন্য সত্তার খোঁজ মেলে।
Hyp: имостиcrosilation inander

Transcribing sonic3_eval_audio_bengali/eval_output_3.wav...
Ref: বর্ষার শেষে মেঘলা আকাশে হঠাৎ এক ঝাঁক সাদা বক আর সবুজ ঘাসের ওপর ব্যাঙের ডাক শোনা গেল।
Hyp: बर्षार सेसे मेगला आकासे, हटत एक जहाक, सादभक और सबूझ गासेर उपर बैंगेड डाक सोना गळलो।

Transcribing sonic3_eval_audio_bengali/eval_output_4.wav...
Ref: সে রেগে গিয়ে বলল, "আ

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
!pip install gnani-vachana jiwer

In [9]:
import json
import os
from jiwer import wer, cer
from gnani.stt import GnaniSTTClient
from google.colab import userdata

# 1. Initialize the Gnani STT Client securely using Colab Secrets
# 1. Initialize the Gnani STT Client
print("Connecting to Gnani.ai API...")
client = GnaniSTTClient(
    api_key="vach_1ytE2CY5X2EPdZgZlDBbi39qeHPJzxX3MWAYEyvf82f5RT6BFzQTTV2PTv3VDgvT3fs2nA5kMEpprB9ea3kjYJO7T13GcGJf_d45996d1e60a19972a97486d5c5955d3"
)

# 2. Load your reference text from the Bengali JSON
input_json = "bengali_evaluation_set.json"
audio_dir = "sonic3_eval_audio_bengali"

with open(input_json, "r", encoding="utf-8") as f:
    data = json.load(f)

references = []
hypotheses = []

print("\nStarting Gnani.ai ASR transcription and evaluation...")

# 3. Iterate through the JSON and transcribe the matching audio
for index, item in enumerate(data):
    reference_text = item.get("bengali_sentence", "")

    if not reference_text:
        continue

    audio_path = os.path.join(audio_dir, f"eval_output_{index}.wav")

    if not os.path.exists(audio_path):
        print(f"Skipping item {index} - Audio file not found.")
        continue

    print(f"Transcribing {audio_path} with Gnani...")

    try:
        # Call the Gnani REST API (Explicitly declaring Bengali-India)
        result = client.transcribe(audio_path, language_code="bn-IN")

        # Extract the transcribed text from the response payload
        hypothesis_text = result.get("transcript", "").strip()

        references.append(reference_text)
        hypotheses.append(hypothesis_text)

        print(f"Ref: {reference_text}")
        print(f"Hyp: {hypothesis_text}\n")

    except Exception as e:
        print(f"Failed to transcribe item {index} via Gnani API: {e}\n")

# 4. Calculate WER and CER
if references and hypotheses:
    overall_wer = wer(references, hypotheses)
    overall_cer = cer(references, hypotheses)

    print("-" * 30)
    print("🏆 GNANI.AI ASR EVALUATION RESULTS 🏆")
    print("-" * 30)
    print(f"Word Error Rate (WER):      {overall_wer:.4f} ({overall_wer * 100:.2f}%)")
    print(f"Character Error Rate (CER): {overall_cer:.4f} ({overall_cer * 100:.2f}%)")
    print("-" * 30)
else:
    print("No audio files were successfully evaluated. Check your credentials and directory.")

Connecting to Gnani.ai API...

Starting Gnani.ai ASR transcription and evaluation...
Transcribing sonic3_eval_audio_bengali/eval_output_0.wav with Gnani...
Ref: অন্ধকার ঘরে কাঁপা হাতে সে পুরোনো পুঁথি আর রথের ভাঙা চাকা খুঁজছে।
Hyp: অন্ধকার ঘরে কাঁপা হাতে সে পুরনো পুঁথি আর রথের ভাঙা চাকা খুঁজছে

Transcribing sonic3_eval_audio_bengali/eval_output_1.wav with Gnani...
Ref: ফাল্গুনের মেলায় ভণ্ড সাধুর কথায় ঘণ্টা বাজতেই এক অদ্ভুত কম্পন তৈরি হলো।
Hyp: ফাল্গুনের মেলায় ভণ্ড সাধুর কথায় ঘণ্টা বাজতেই এক অদ্ভুত কম্পন তৈরি হল

Transcribing sonic3_eval_audio_bengali/eval_output_2.wav with Gnani...
Ref: শহরের এই উচ্চ অট্টালিকার ছাদে দাঁড়ালে উদ্দাম বাতাসের শব্দে অন্য সত্তার খোঁজ মেলে।
Hyp: শহরের এই উচ্চ অট্টালিকার ছাদে দাঁড়ালে উদ্দাম বাতাসের শব্দে অন্য সত্তার খোঁজ মেলে

Transcribing sonic3_eval_audio_bengali/eval_output_3.wav with Gnani...
Ref: বর্ষার শেষে মেঘলা আকাশে হঠাৎ এক ঝাঁক সাদা বক আর সবুজ ঘাসের ওপর ব্যাঙের ডাক শোনা গেল।
Hyp: বর্ষার শেষে মেঘলা আকাশে হঠাৎ এক ঝাঁক সাদা বক আর সবুজ ঘাসের ওপর ব্যাঙে